In [1]:
import pandas as pd

In [2]:
!pip install psycopg2-binary pandas sqlalchemy
import psycopg2

In [3]:
from sqlalchemy import create_engine

# Step 1: Connect to PostgreSQL
dbname = 'online_payment_fraud_detection'
user = 'postgres'
password = 'Postgresql1999%40' #Pw is 'Postgresql1999@' use %40 in place of @ to not get any error in connection_string
host = 'localhost' # or remote host IP
port = '5432' # default port

# Create the connection string for SQLAlchemy
connection_string = f"postgresql+psycopg2://{user}:{password}@{host}:{port}/{dbname}"
engine = create_engine(connection_string)

In [4]:
# Write your SQL query here
query = "SELECT * FROM transactions;"

# Load the data into a DataFrame
df = pd.read_sql(query, con=engine)

# Display the first few rows
print(df.head(5))


   step     type     amount      nameorg  oldbalanceorg  newbalanceorg  \
0     7  CASH_IN   20042.90   C687328420     4907313.55     4927356.46   
1     7  CASH_IN  137054.90  C1612040005     4927356.46     5064411.36   
2     7  CASH_IN  113656.68   C760487425     5064411.36     5178068.04   
3     7  CASH_IN   65256.48  C1284076567     5178068.04     5243324.52   
4     7  CASH_IN  222536.02  C1519190990     5243324.52     5465860.55   

      namedest  oldbalancedest  newbalancedest  isfraud  isflaggedfraud  \
0    C36856762       500713.55       480670.65        0               0   
1  C1112414583       143748.04         6693.13        0               0   
2  C1636786811      6849235.33      6779300.28        0               0   
3   C931482420       112891.15        47634.67        0               0   
4   C347725669       708601.42       266931.55        0               0   

   balancedifforg  balancediffdest  hourofday  
0       -20042.91        -20042.90          7  
1      -

In [5]:
df.columns

Index(['step', 'type', 'amount', 'nameorg', 'oldbalanceorg', 'newbalanceorg',
       'namedest', 'oldbalancedest', 'newbalancedest', 'isfraud',
       'isflaggedfraud', 'balancedifforg', 'balancediffdest', 'hourofday'],
      dtype='object')

In [6]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

In [7]:
# Encode categorical column
le = LabelEncoder()
df['type_encoded'] = le.fit_transform(df['type'])

In [8]:
# Select features and target
features = ['amount', 'balancedifforg', 'balancediffdest', 'hourofday', 'type_encoded']
X = df[features]
y = df['isfraud']

In [9]:
# Train-test split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)


In [10]:
# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [11]:
# Logistic Regression
lr = LogisticRegression()
lr.fit(X_train_scaled, y_train)
y_pred_lr = lr.predict(X_test_scaled)

print("Logistic Regression Results:")
print(confusion_matrix(y_test, y_pred_lr))
print(classification_report(y_test, y_pred_lr))
print("ROC AUC:", roc_auc_score(y_test, lr.predict_proba(X_test_scaled)[:,1]))


Logistic Regression Results:
[[1906104     218]
 [   1351    1113]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906322
           1       0.84      0.45      0.59      2464

    accuracy                           1.00   1908786
   macro avg       0.92      0.73      0.79   1908786
weighted avg       1.00      1.00      1.00   1908786

ROC AUC: 0.9615864621820134


In [12]:
# Random Forest
rf = RandomForestClassifier(n_estimators=10, random_state=42)
rf.fit(X_train, y_train)
y_pred_rf = rf.predict(X_test)

print("\nRandom Forest Results:")
print(confusion_matrix(y_test, y_pred_rf))
print(classification_report(y_test, y_pred_rf))
print("ROC AUC:", roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))


Random Forest Results:
[[1906142     180]
 [    556    1908]]
              precision    recall  f1-score   support

           0       1.00      1.00      1.00   1906322
           1       0.91      0.77      0.84      2464

    accuracy                           1.00   1908786
   macro avg       0.96      0.89      0.92   1908786
weighted avg       1.00      1.00      1.00   1908786

ROC AUC: 0.9158100213914679
